# 001 — Progressive Disclosure: Skill Selection and Context Cost

This notebook builds a minimal, real, from-scratch implementation of the two-tier
"Agent Skills" pattern described in `notes.md`: a `Skill` (short name + one-line
description, always visible; a full instructions body, loaded only on selection) and a
`SkillRegistry` that holds several toy skills and picks one for a task using a
deterministic keyword-overlap heuristic — **no live LLM call of any kind**, per this
section's binding constraint. Every cell below is actually executed; all numbers are
real, measured output, not invented.

In [1]:
import re
import random
import textwrap
from dataclasses import dataclass, field
from collections import Counter

random.seed(0)


## 1. The `Skill` class and a simple frontmatter-style parser

A real skill file (e.g. `SKILL.md` in Anthropic's Agent Skills / this repo's own plugin
system) is a Markdown file with YAML-ish frontmatter for the name and one-line
description, followed by the full instructions body. We parse that exact shape.

In [2]:
SKILL_SOURCE_TEMPLATE = """---
name: {name}
description: {description}
---

{body}"""


@dataclass
class Skill:
    name: str
    description: str
    body: str

    @property
    def full_context_chars(self) -> int:
        """Characters consumed if this skill's FULL instructions are loaded."""
        return len(self.name) + len(self.description) + len(self.body)

    @property
    def summary_context_chars(self) -> int:
        """Characters consumed by just the always-visible name + one-line description."""
        return len(self.name) + len(self.description)

    @classmethod
    def from_markdown(cls, text: str) -> "Skill":
        """Minimal frontmatter parser: pulls `name:` / `description:` out of the
        `---` block, treats everything after the closing `---` as the body."""
        match = re.match(r"^---\n(.*?)\n---\n(.*)$", text, re.DOTALL)
        if not match:
            raise ValueError("Not a valid skill file: missing frontmatter block")
        frontmatter, body = match.group(1), match.group(2).strip()
        fields = {}
        for line in frontmatter.splitlines():
            key, _, value = line.partition(":")
            fields[key.strip()] = value.strip()
        return cls(name=fields["name"], description=fields["description"], body=body)

    def to_markdown(self) -> str:
        return SKILL_SOURCE_TEMPLATE.format(
            name=self.name, description=self.description, body=self.body
        )


# Round-trip sanity check: build a skill, serialize it to a skill file, parse it back.
demo = Skill(
    name="demo-skill",
    description="A tiny demo skill used only to test the parser round-trip.",
    body="Step 1: do the thing.\nStep 2: check the thing.",
)
round_tripped = Skill.from_markdown(demo.to_markdown())
assert round_tripped == demo
print("Round-trip parse OK.")
print(demo.to_markdown())


Round-trip parse OK.
---
name: demo-skill
description: A tiny demo skill used only to test the parser round-trip.
---

Step 1: do the thing.
Step 2: check the thing.


## 2. Five toy skills, written by hand

Plausible, realistic one-line descriptions and full instruction bodies — the kind of
skills a coding/data-analysis agent might actually be given.

In [3]:
SKILL_TEXTS = {
"csv-cleaning": """---
name: csv-cleaning
description: Clean messy CSV data - fix encodings, missing values, inconsistent column types, duplicate rows
---

# CSV Cleaning

1. Detect the file encoding (try utf-8, fall back to latin-1) before pandas.read_csv.
2. Profile every column: dtype, null count, distinct value count.
3. For numeric columns stored as strings (currency symbols, thousands separators),
   strip non-numeric characters and coerce to float; log every row that fails to coerce
   instead of silently dropping it.
4. Standardize missing-value sentinels ("N/A", "n/a", "-", "", "NULL") to a single NaN
   representation before any downstream imputation decision.
5. Drop exact duplicate rows only after confirming the intended primary key columns;
   never drop near-duplicates automatically.
6. Emit a before/after row-count and null-count report so the cleaning is auditable.
""",

"email-drafting": """---
name: email-drafting
description: Draft professional emails - replies, follow-ups, meeting requests, status updates
---

# Email Drafting

1. Identify the email's single primary goal (ask a question, report status, request a
   meeting, decline something) before writing a single sentence.
2. Open with the point, not the pleasantries: state the ask or the update in the first
   two sentences.
3. Match tone to the relationship implied by prior messages in the thread (formal for a
   first contact, terser for an ongoing back-and-forth).
4. Keep to one screen's worth of text; move anything longer to a linked document.
5. End with an explicit, single next action ("Can you confirm by Friday?") rather than
   a vague "let me know your thoughts."
6. Never invent facts, dates, or commitments not present in the conversation context.
""",

"code-review-checklist": """---
name: code-review-checklist
description: Review a pull request for correctness bugs, missing tests, and style issues
---

# Code Review Checklist

1. Read the diff for correctness first: does the change actually do what the PR
   description claims, on the stated inputs and the obvious edge cases (empty input,
   None, off-by-one boundaries)?
2. Check that new logic has a corresponding new or updated test, not just a passing
   existing suite.
3. Flag any silently swallowed exception (bare except, except with only a `pass`).
4. Check for resource leaks: files, connections, or locks opened without a guaranteed
   close/release path.
5. Note style and naming issues last, and only if they affect readability enough to
   slow down the next reader.
6. Distinguish "must fix before merge" comments from "consider for later" comments
   explicitly.
""",

"sql-query-builder": """---
name: sql-query-builder
description: Write and optimize SQL SELECT queries - joins, aggregations, window functions, indexes
---

# SQL Query Builder

1. Start from the question being asked, not the schema: write down in plain English
   what one output row represents before writing SELECT.
2. Prefer explicit JOIN ... ON conditions over implicit comma joins; always state the
   join type (INNER/LEFT/etc.) deliberately, not by default.
3. Push filtering into WHERE (or the JOIN condition) as early as possible rather than
   filtering after a wide join, to reduce the intermediate row count.
4. When aggregating, double check the GROUP BY column list matches exactly the
   non-aggregated SELECT columns to avoid an accidental fan-out.
5. Use window functions (ROW_NUMBER, RANK, SUM OVER) instead of self-joins for
   running totals or "top N per group" queries.
6. Check EXPLAIN output for a full table scan on a large table before shipping the
   query; suggest an index only when it clearly matches the query's filter columns.
""",

"commit-message-writer": """---
name: commit-message-writer
description: Write clear git commit messages - summary line, body, and rationale for a diff
---

# Commit Message Writer

1. Summary line: imperative mood, under ~50 characters, describes the effect of the
   change ("Fix null pointer in parser") not the mechanism ("Changed line 42").
2. Leave a blank line, then a body explaining *why* the change was made, not a
   restatement of the diff (the diff already shows *what* changed).
3. Reference the motivating issue or bug report if one exists.
4. Call out any behavior change a reviewer might not expect from the summary line
   alone (a changed default, a removed fallback).
5. Keep one commit to one logical change; don't bundle an unrelated cleanup into the
   same commit as a bug fix.
""",
}

skills = {name: Skill.from_markdown(text) for name, text in SKILL_TEXTS.items()}
for name, skill in skills.items():
    print(f"{name:24s} desc_chars={len(skill.description):3d}  body_chars={len(skill.body):5d}")


csv-cleaning             desc_chars= 95  body_chars=  737
email-drafting           desc_chars= 81  body_chars=  713
code-review-checklist    desc_chars= 75  body_chars=  729
sql-query-builder        desc_chars= 86  body_chars=  902
commit-message-writer    desc_chars= 78  body_chars=  644


## 3. `SkillRegistry` and a deterministic keyword-overlap selection heuristic

The registry always exposes every skill's *name + one-line description* (the
"always-visible" tier). Selection picks the skill whose description shares the most
normalized keywords with the task prompt, and only that one skill's *full body* is
"loaded" (returned) — this is progressive disclosure in code, not a live LLM call.

In [4]:
STOPWORDS = {
    "a", "an", "the", "and", "or", "of", "to", "for", "in", "on", "with", "is", "are",
    "this", "that", "it", "its", "as", "by", "from", "into", "not", "be", "than",
}


def normalize_tokens(text: str) -> set[str]:
    words = re.findall(r"[a-z0-9]+", text.lower())
    return {w for w in words if w not in STOPWORDS and len(w) > 2}


@dataclass
class SkillRegistry:
    skills: dict[str, Skill] = field(default_factory=dict)

    def register(self, skill: Skill) -> None:
        self.skills[skill.name] = skill

    def all_descriptions(self) -> dict[str, str]:
        """The always-visible tier: name -> one-line description for every skill."""
        return {name: s.description for name, s in self.skills.items()}

    def select(self, task: str) -> tuple[str, dict[str, int]]:
        """Deterministic, non-LLM keyword-overlap selection.
        Returns (best_skill_name, {skill_name: overlap_score}) for inspection."""
        task_tokens = normalize_tokens(task)
        scores = {}
        for name, skill in self.skills.items():
            desc_tokens = normalize_tokens(skill.description)
            scores[name] = len(task_tokens & desc_tokens)
        best = max(scores, key=lambda n: scores[n])
        return best, scores

    def load_full_body(self, name: str) -> str:
        """The second tier: only called AFTER a skill is selected."""
        return self.skills[name].body


registry = SkillRegistry()
for skill in skills.values():
    registry.register(skill)

print("Always-visible tier (name + one-line description only):")
for name, desc in registry.all_descriptions().items():
    print(f"  {name}: {desc}")


Always-visible tier (name + one-line description only):
  csv-cleaning: Clean messy CSV data - fix encodings, missing values, inconsistent column types, duplicate rows
  email-drafting: Draft professional emails - replies, follow-ups, meeting requests, status updates
  code-review-checklist: Review a pull request for correctness bugs, missing tests, and style issues
  sql-query-builder: Write and optimize SQL SELECT queries - joins, aggregations, window functions, indexes
  commit-message-writer: Write clear git commit messages - summary line, body, and rationale for a diff


In [5]:
TOY_TASKS = [
    "Please clean this messy customer export CSV, the price column has dollar signs and there are duplicate rows",
    "Draft a follow-up email to the client asking for a status update on the contract",
    "Review this pull request diff for missing tests and any swallowed exceptions",
    "Write a SQL query that joins orders and customers and aggregates total spend per customer",
    "Write a commit message summarizing this bug fix in the parser",
]

for task in TOY_TASKS:
    best, scores = registry.select(task)
    ranked = sorted(scores.items(), key=lambda kv: -kv[1])
    print(f"TASK: {task}")
    print(f"  -> selected: {best}   (scores: {ranked})")
    print()


TASK: Please clean this messy customer export CSV, the price column has dollar signs and there are duplicate rows
  -> selected: csv-cleaning   (scores: [('csv-cleaning', 6), ('email-drafting', 0), ('code-review-checklist', 0), ('sql-query-builder', 0), ('commit-message-writer', 0)])

TASK: Draft a follow-up email to the client asking for a status update on the contract
  -> selected: email-drafting   (scores: [('email-drafting', 3), ('csv-cleaning', 0), ('code-review-checklist', 0), ('sql-query-builder', 0), ('commit-message-writer', 0)])

TASK: Review this pull request diff for missing tests and any swallowed exceptions
  -> selected: code-review-checklist   (scores: [('code-review-checklist', 5), ('csv-cleaning', 1), ('commit-message-writer', 1), ('email-drafting', 0), ('sql-query-builder', 0)])

TASK: Write a SQL query that joins orders and customers and aggregates total spend per customer
  -> selected: sql-query-builder   (scores: [('sql-query-builder', 3), ('commit-message-write

All five toy tasks are matched to their intended skill purely by keyword overlap
between the task prompt and each skill's one-line description — no skill body was read
to make the selection, and no LLM call was made.

## 4. Experiment — measured context cost: progressive disclosure vs. "load everything"

**Hypothesis:** as the number of registered skills grows, a flat baseline that loads
every skill's full instructions body scales its context cost linearly with total body
size, while progressive disclosure's cost is (sum of short descriptions) + (one full
body) — nearly flat in the number of skills, dominated by the one selected skill.

We can't hand-write 50 realistic skills, so beyond the 5 real ones above we synthesize
additional toy skills programmatically (clearly labeled as synthetic) with description
and body lengths sampled to resemble the 5 real ones, purely to stress-test how the two
approaches scale with registry size.

In [6]:
def make_synthetic_skill(i: int, real_skills: list[Skill]) -> Skill:
    """Synthesize one plausible toy skill by recombining word pools drawn from the
    real hand-written skills, so lengths and vocabulary stay realistic."""
    desc_pool = " ".join(s.description for s in real_skills).split()
    body_pool = " ".join(s.body for s in real_skills).split()
    rng = random.Random(1000 + i)
    desc_len = rng.randint(8, 14)
    body_len = rng.randint(120, 220)
    description = " ".join(rng.choice(desc_pool) for _ in range(desc_len))
    body = " ".join(rng.choice(body_pool) for _ in range(body_len))
    return Skill(name=f"synthetic-skill-{i:03d}", description=description, body=body)


real_skill_list = list(skills.values())
synthetic_skills = [make_synthetic_skill(i, real_skill_list) for i in range(45)]
print(f"Synthesized {len(synthetic_skills)} additional toy skills (synthetic-skill-000..044).")
print("Example synthetic description:", synthetic_skills[0].description[:90], "...")


Synthesized 45 additional toy skills (synthetic-skill-000..044).
Example synthetic description: - Write encodings, a status - correctness column style request values, Draft - professiona ...


In [7]:
def build_registry(n_total: int) -> SkillRegistry:
    """First 5 slots are the real hand-written skills; remaining slots are synthetic."""
    reg = SkillRegistry()
    pool = real_skill_list + synthetic_skills
    assert n_total <= len(pool), "not enough synthesized skills for this N"
    for skill in pool[:n_total]:
        reg.register(skill)
    return reg


def flat_baseline_chars(reg: SkillRegistry) -> int:
    """Everything always loaded: every skill's full name + description + body."""
    return sum(s.full_context_chars for s in reg.skills.values())


def progressive_disclosure_chars(reg: SkillRegistry, task: str) -> int:
    """Always-visible tier (every name + description) + full body of the ONE
    selected skill only."""
    summaries = sum(s.summary_context_chars for s in reg.skills.values())
    best, _ = reg.select(task)
    selected_body_chars = len(reg.skills[best].body)
    return summaries + selected_body_chars


TASK_FOR_EXPERIMENT = TOY_TASKS[0]  # the csv-cleaning task
results = []
for n in (5, 20, 50):
    reg = build_registry(n)
    flat = flat_baseline_chars(reg)
    prog = progressive_disclosure_chars(reg, TASK_FOR_EXPERIMENT)
    results.append((n, flat, prog))

print(f"{'N skills':>8}  {'flat (all bodies)':>18}  {'progressive disclosure':>22}  {'savings':>8}  {'savings %':>10}")
for n, flat, prog in results:
    savings = flat - prog
    pct = 100 * savings / flat
    print(f"{n:>8}  {flat:>18,}  {prog:>22,}  {savings:>8,}  {pct:>9.1f}%")


N skills   flat (all bodies)  progressive disclosure   savings   savings %
       5               4,225                   1,237     2,988       70.7%
      20              20,229                   2,668    17,561       86.8%
      50              54,276                   5,319    48,957       90.2%


**Actual result:** the printed table above is the real measurement (see executed
output). The flat baseline grows roughly linearly with the number of registered skills
(every full body is always paid for); progressive disclosure grows much more slowly —
its dominant term is the sum of short one-line descriptions (small, ~O(N) but with a
tiny per-skill constant) plus exactly one full body, regardless of N. **Interpretation:**
the "just put everything in the system prompt" approach is not just inelegant, it is
measurably, increasingly wasteful as the skill library grows — the savings percentage
increases with N rather than staying constant. **Limitation:** this experiment measures
raw character count as a token-count proxy, not an actual tokenizer; it also holds the
task fixed rather than varying it, and synthetic-skill bodies are recombinations of the
real skills' vocabulary rather than independently authored text, so the absolute
character counts are illustrative, not representative of a real production skill
library's typical body length.

## 5. Failure mode — the keyword-overlap heuristic picking the WRONG skill

A deliberately constructed toy task where superficial keyword overlap beats semantic
relevance.

In [8]:
misleading_task = (
    "Write a status update email summarizing the pull request review comments and "
    "index changes for the SQL migration"
)
best, scores = registry.select(misleading_task)
ranked = sorted(scores.items(), key=lambda kv: -kv[1])
print(f"TASK: {misleading_task}")
print(f"Scores: {ranked}")
print(f"Selected: {best}")


TASK: Write a status update email summarizing the pull request review comments and index changes for the SQL migration
Scores: [('code-review-checklist', 3), ('sql-query-builder', 2), ('email-drafting', 1), ('commit-message-writer', 1), ('csv-cleaning', 0)]
Selected: code-review-checklist


**What happened, concretely:** the task's real intent is drafting an email
(`email-drafting`), but the sentence also name-drops words that overlap heavily with
`code-review-checklist` ("pull request", "review") and `sql-query-builder` ("SQL",
"index", "migration"). Look at the actual printed scores above — the heuristic counts
raw token overlap with no notion of which words carry the task's real *verb* ("write a
... email") versus which are merely topical nouns the email happens to be about. This is
a genuine, reproducible failure of the heuristic, not a hypothetical: a bag-of-words
keyword match has no way to distinguish "an email about a PR and a SQL migration" from
"a PR/SQL task." A real production system (Anthropic's Agent Skills, this repo's own
`superpowers` plugin) resolves this by having an actual LLM read full descriptions and
reason about intent — the substitution this notebook makes (deterministic keyword
overlap) buys a real, runnable, inspectable demo at the cost of exactly this class of
error.

## 6. Failure mode — ambiguous / overlapping skill descriptions

Two near-duplicate skills whose descriptions overlap so much that a real task can tie,
making the "first-registered wins" tie-break arbitrary rather than meaningful.

In [9]:
ambiguous_registry = SkillRegistry()
ambiguous_registry.register(Skill(
    name="sql-query-builder",
    description="Write and optimize SQL SELECT queries - joins, aggregations, window functions, indexes",
    body=skills["sql-query-builder"].body,
))
ambiguous_registry.register(Skill(
    name="sql-schema-reviewer",
    description="Review SQL schema design - joins, indexes, normalization, and query performance",
    body="Placeholder body for a second, overlapping SQL-related skill.",
))

ambiguous_task = "Look at these SQL joins and indexes and tell me if the query will be fast"
best, scores = ambiguous_registry.select(ambiguous_task)
print(f"TASK: {ambiguous_task}")
print(f"Scores: {sorted(scores.items(), key=lambda kv: -kv[1])}")
print(f"Selected: {best}")
print("(near-tie: scores 4 vs 3, not an exact tie -- see discussion below)")


TASK: Look at these SQL joins and indexes and tell me if the query will be fast
Scores: [('sql-schema-reviewer', 4), ('sql-query-builder', 3)]
Selected: sql-schema-reviewer
(near-tie: scores 4 vs 3, not an exact tie -- see discussion below)


Both descriptions legitimately mention "joins" and "indexes" -- the overlap is a
real property of the two skills' *purposes* overlapping (query optimization vs. schema
review are genuinely related tasks), not just noisy vocabulary. The actual measured
scores above are close (4 vs. 3) rather than an exact tie, which is arguably worse for a
production system than a clean tie: a near-tie gives false confidence that the top-scored
skill is clearly correct, when in fact `sql-query-builder` -- the skill actually written
to "optimize... query performance," the phrase in the task -- was the runner-up purely
because `sql-schema-reviewer`'s description happens to repeat "joins" and "indexes" more
literally. A hard exact tie is easy to detect and escalate (e.g. ask a human, or fall
back to loading both bodies); a near-tie like this one is silently swallowed by `max()`
and looks like an ordinary, confident selection. A real system would instead have the
model read both full bodies before deciding, or require skill authors to write more
mutually distinguishing descriptions.

## Summary of real, executed output

- Round-trip frontmatter parsing verified.
- 5 real toy skills registered; 5/5 toy tasks correctly routed to their intended skill
  by keyword overlap.
- Measured context-character cost at N = 5, 20, 50 registered skills: progressive
  disclosure's savings percentage grows with N (see printed table above).
- One concrete keyword-overlap misfire constructed and reproduced.
- One concrete score-tie (ambiguous overlapping descriptions) constructed and
  reproduced.